# PlantVision 2.0 — Houseplant disease pipeline

**Stage 1:** YOLOv8n single-class `plant_leaf` detector, trained on **300–800 hand-labelled indoor leaf bboxes** plus targeted Roboflow Universe houseplant slices. PlantDoc is **not** used (its agricultural close-ups bias localisation toward field shots and hurt indoor recall).

**Stage 2:** EfficientNetV2-S teacher → distilled MobileNetV3-Large student. 11 final classes:

1. `bacterial_leaf_spot`
2. `edema`
3. `gray_mold_botrytis`
4. `healthy`
5. `leaf_scorch_underwatering`
6. `leaf_spot_fungal` (anthracnose + cercospora + alternaria)
7. `not_a_plant`
8. `nutrient_deficiency` (Fe + N + Mg + K)
9. `powdery_mildew`
10. `sunburn_or_scorch`
11. `viral_disease` (mosaic + leaf_curl + ringspot)

Removed vs v1.x: `late_blight`, `iron_chlorosis` (subsumed). Pests are out of the taxonomy entirely.

**Primary metric:** disease macro-F1 on **`indoor_real_world_v1`** — a frozen 500–1000 photo validation set captured by real users on different devices, windows, lamps and distances. Without it, top-1 on a quirky train slice predicts nothing about app behaviour.

**PlantVillage** stays in the project only as an **isolated `val_studio` slice** (transfer-metric + background-bias monitor — Noyan 2022 showed 49% acc from 8 background pixels). It is never in the train set.

**Time budget on 1× A100-40:**
- 5–10 min DINOv2-S frozen + linear probe (diagnostic)
- ~35 min EfficientNetV2-S fine-tune (teacher)
- 10–15 min KD into MobileNetV3-Large student
- 5 min temperature calibration + ONNX export

## 1. Environment

`timm` for the V2-S backbone; `cleanlab` + `fiftyone` for the data-cleanup pass; `imagehash` for pHash dedup.

In [ ]:
import sys, subprocess
PIP = [sys.executable, '-m', 'pip', 'install', '-q']
PKGS = [
    'torch>=2.2', 'torchvision', 'timm>=1.0',
    'ultralytics>=8.2', 'onnx', 'onnxruntime',
    'cleanlab>=2.6', 'fiftyone>=0.24',
    'imagehash', 'Pillow', 'numpy', 'scikit-learn', 'pandas',
    'opencv-python-headless', 'tqdm', 'matplotlib',
]
subprocess.check_call(PIP + PKGS)

## 2. Hierarchical annotation schema

Every cropped leaf carries the metadata below as a sidecar JSON, so error-analysis joins (group by `source`, `capture_session`, etc.) stay cheap even though the model is single-head.

Sources are kept disjoint at `plant_id` level for splits — the same plant must never appear in train and val. `capture_session` lets us drop near-duplicates from the same shoot.

In [ ]:
from dataclasses import dataclass, asdict
from typing import Optional, List
import json, pathlib

@dataclass
class CropAnnotation:
    plant_id: str
    species: Optional[str]
    diagnosis: str
    severity: str               # low | medium | high
    leaf_or_whole_plant: str    # leaf | whole_plant
    capture_session: str
    source: str                 # user_upload | inat | roboflow | money_atlas | oe_pool
    label_confidence: float     # in [0, 1]
    bbox: List[int]             # [x, y, w, h]

def write_sidecar(crop_path: pathlib.Path, ann: CropAnnotation):
    crop_path.with_suffix('.json').write_text(
        json.dumps(asdict(ann), ensure_ascii=False, indent=2)
    )

## 3. Data sources (NO PlantVillage in train)

* **Money Plant Disease Atlas** (Biswas 2024, ~3k Pothos crops).
* **Roboflow Universe** — point-pick houseplant disease projects only.
* **iNaturalist** — `quality_grade=research` + `field:plant_part_observed=leaf`. Use the correct **taxon IDs** for indoor genera (Ficus, Monstera, Pothos, Calathea, Spathiphyllum, Dracaena, etc.). **Drop Phytophthora cinnamomi observations** — root pathogen, no leaf signal.
* **PlantVillage** — only into `val_studio` (frozen transfer / background-bias monitor). **Never train.**
* **`indoor_real_world_v1`** — 500–1000 user photos with the schema from §2. Holdout split is the **primary metric** target.
* **OE-pool for `not_a_plant`** — filtered OpenImages V7 indoor scenes, iNat Animalia, COCO val2017, future kicked-out user photos. Do NOT download the full OpenImages tarball (hundreds of GB).

### Three independent validation slices

| Slice | Source | Used for | Selection rule |
|---|---|---|---|
| `val_indomain` | random 10–15 % holdout of `train/` | early stopping, schedule decisions | group-aware split by `plant_id` |
| `val_studio`   | `plantvillage_studio` slice | studio-transfer + background-bias monitor (Noyan 2022) | never enters train |
| `val_realworld`| `indoor_real_world_v1` user holdout | **primary metric — ship/no-ship gate** | frozen at v2.0; new photos go to a separate v2.1 slice |

The shipping decision is on `val_realworld` macro-F1 over the disease classes (excludes `healthy` and `not_a_plant`). Target: **0.70 – 0.78**. `val_indomain` is for tuning loops only; `val_studio` is a transfer monitor. Each epoch we log macro-F1 + per-class recall + confusion matrix on **all three** so a healthy collapse or rare-class death is visible the moment it happens.


In [ ]:
import pathlib

ROOT = pathlib.Path('data_v2')
TRAIN_ROOT = ROOT / 'train'                  # 11 class subfolders
VAL_INDOMAIN = ROOT / 'val' / 'indomain'                       # group-aware holdout from train, drives early stopping
VAL_PRIMARY = ROOT / 'val' / 'indoor_real_world_v1'   # PRIMARY — ship/no-ship gate
VAL_STUDIO  = ROOT / 'val' / 'plantvillage_studio'   # isolated transfer slice
OE_POOL     = ROOT / 'oe_pool'

CLASSES = [
    'bacterial_leaf_spot', 'edema', 'gray_mold_botrytis', 'healthy',
    'leaf_scorch_underwatering', 'leaf_spot_fungal', 'not_a_plant',
    'nutrient_deficiency', 'powdery_mildew', 'sunburn_or_scorch', 'viral_disease',
]
for c in CLASSES:
    (TRAIN_ROOT / c).mkdir(parents=True, exist_ok=True)
VAL_INDOMAIN.mkdir(parents=True, exist_ok=True)
VAL_PRIMARY.mkdir(parents=True, exist_ok=True)
VAL_STUDIO.mkdir(parents=True, exist_ok=True)
OE_POOL.mkdir(parents=True, exist_ok=True)
print('Layout ready under', ROOT)

### 3a. Eval helpers — three independent val loaders

Build one DataLoader per slice and a single `evaluate()` helper that emits the same metric tuple (macro-F1 over diseases, per-class recall, healthy-bias monitor, confusion matrix). The training loop calls it every epoch on **all three** so we never ship a model that healed `val_indomain` while collapsing `val_realworld`.

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms as T
import numpy as np
from sklearn.metrics import f1_score, recall_score, confusion_matrix

NORM = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

def make_eval_loader(root, img_size, batch=128, workers=4):
    """Deterministic eval transform; no augs, no shuffling."""
    tx = T.Compose([
        T.Resize(int(img_size * 1.05)),
        T.CenterCrop(img_size),
        T.ToTensor(),
        NORM,
    ])
    ds = ImageFolder(str(root), transform=tx)
    dl = DataLoader(ds, batch_size=batch, shuffle=False,
                    num_workers=workers, pin_memory=True)
    return ds, dl

@torch.inference_mode()
def evaluate(model, dl, class_names, device, name=""):
    """Return macro-F1 over diseases + per-class recall + confusion matrix.

    Macro-F1 excludes healthy and not_a_plant — those classes are huge
    and bias the average. Per-class recall surfaces rare-class collapse
    (model giving up on edema / viral_disease) the moment it happens."""
    model.eval()
    y_true, y_pred = [], []
    for x, y in dl:
        x = x.to(device, non_blocking=True)
        logits = model(x)
        y_pred.extend(logits.argmax(1).cpu().tolist())
        y_true.extend(y.tolist())
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    disease_idx = [i for i, n in enumerate(class_names)
                   if n not in ("healthy", "not_a_plant")]
    macro_f1_disease = f1_score(
        y_true, y_pred, labels=disease_idx, average="macro", zero_division=0,
    )
    per_class_recall = recall_score(
        y_true, y_pred, labels=list(range(len(class_names))),
        average=None, zero_division=0,
    )
    healthy_idx = class_names.index("healthy") if "healthy" in class_names else None
    healthy_pred_share = float((y_pred == healthy_idx).mean()) if healthy_idx is not None else 0.0
    healthy_true_share = float((y_true == healthy_idx).mean()) if healthy_idx is not None else 0.0
    healthy_bias = (healthy_pred_share / max(healthy_true_share, 1e-9)) if healthy_idx is not None else 0.0
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    print(f"[{name}] disease_macro_f1={macro_f1_disease:.3f}  "
          f"healthy_bias={healthy_bias:.2f}  n={len(y_true)}")
    for cls, r in zip(class_names, per_class_recall):
        print(f"    recall[{cls:30s}] = {r:.3f}")
    return {
        "name": name,
        "disease_macro_f1": float(macro_f1_disease),
        "per_class_recall": {c: float(r) for c, r in zip(class_names, per_class_recall)},
        "healthy_bias": float(healthy_bias),
        "confusion_matrix": cm.tolist(),
    }


## 4. Dedup & label-noise cleanup (before any training)

Cleanup ROI is higher than any scheduler tweak. Pipeline:

1. **SHA-256** dedup — exact byte-equal copies.
2. **pHash** (`imagehash.phash`, Hamming ≤ 8) — near-duplicates from re-encoded posts.
3. **DINOv2 ViT-B/14** embeddings + per-class kNN: drop crops > 2σ from class centroid.
4. Manual validation pass on **200–300 photos per class**.
5. **Cleanlab** label-issue ranking + **FiftyOne** embedding browser to triage top-N suspects.

In [ ]:
import hashlib, imagehash
from PIL import Image
import numpy as np

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

def dedup_directory(root, phash_threshold=8):
    seen_sha, seen_phash = set(), []
    keep, drop = [], []
    for p in sorted(root.rglob('*')):
        if not p.is_file() or p.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.webp'}:
            continue
        s = sha256_file(p)
        if s in seen_sha:
            drop.append((p, 'sha256_dup')); continue
        seen_sha.add(s)
        try:
            ph = imagehash.phash(Image.open(p).convert('RGB'))
        except Exception:
            drop.append((p, 'unreadable')); continue
        if any((ph - other) <= phash_threshold for other in seen_phash):
            drop.append((p, 'phash_dup')); continue
        seen_phash.append(ph)
        keep.append(p)
    return keep, drop

In [ ]:
import torch
from torchvision import transforms

def dinov2_embeddings(image_paths, device='cuda'):
    model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14').eval().to(device)
    tx = transforms.Compose([
        transforms.Resize(252, transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    embs = []
    with torch.inference_mode():
        for p in image_paths:
            x = tx(Image.open(p).convert('RGB')).unsqueeze(0).to(device)
            embs.append(model(x).cpu().numpy().squeeze())
    return np.stack(embs)

def drop_outliers_per_class(embs: np.ndarray, sigma: float = 2.0):
    centroid = embs.mean(0)
    d = np.linalg.norm(embs - centroid, axis=1)
    return d <= (d.mean() + sigma * d.std())

# Cleanlab + FiftyOne triage (pseudo-code — wire to probe outputs from §6)
# from cleanlab.rank import get_label_quality_scores
# scores = get_label_quality_scores(labels=y_true, pred_probs=probs)
# suspects = np.argsort(scores)[:200]
# import fiftyone as fo, fiftyone.brain as fob
# ds = fo.Dataset.from_dir(str(TRAIN_ROOT), dataset_type=fo.types.ImageClassificationDirectoryTree)
# fob.compute_visualization(ds, embeddings=embs, brain_key='dinov2_emb')
# fo.launch_app(ds)

## 5. Stage 1 — YOLOv8n single-class `plant_leaf` (indoor)

Train on the 300–800 hand-labelled indoor leaf bboxes (single class). Roboflow Universe houseplant projects are the supplementary source. PlantDoc excluded by design.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
model.train(
    data='data_v2/yolo/plant_leaf_indoor.yaml',  # 1 class: plant_leaf
    epochs=60, patience=12, batch=32, imgsz=640,
    optimizer='AdamW', lr0=1e-3, cos_lr=True, amp=True,
    mosaic=0.5, hsv_h=0.0, hsv_s=0.4, hsv_v=0.4,
    project='runs/detect', name='plant_leaf_indoor_v1',
)
model.export(format='onnx', opset=13, dynamic=False)

## 6. Diagnostic baseline — DINOv2-S frozen + linear probe (5–10 min)

Mandatory before tweaking anything else.

* Probe ≫ student → bottleneck is **data** (clean it, change sources).
* Probe ≈ chance → bottleneck is **labels / semantics** (revisit class definitions).
* Probe ≈ student → bottleneck is **capacity / augs** (now scheduler tweaks pay off).

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms as T

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tx_eval = T.Compose([
    T.Resize(252, T.InterpolationMode.BICUBIC), T.CenterCrop(224),
    T.ToTensor(), T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])
train_ds = ImageFolder(str(TRAIN_ROOT), transform=tx_eval)
val_ds   = ImageFolder(str(VAL_PRIMARY), transform=tx_eval)
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=4, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=256, shuffle=False, num_workers=4, pin_memory=True)

backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').eval().to(device)
for p in backbone.parameters(): p.requires_grad_(False)
head = nn.Linear(384, len(CLASSES)).to(device)
opt = torch.optim.AdamW(head.parameters(), lr=3e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)

for ep in range(8):
    head.train()
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device)
        with torch.no_grad():
            f = backbone(x)
        opt.zero_grad(); loss_fn(head(f), y).backward(); opt.step()
    head.eval(); correct = total = 0
    with torch.inference_mode():
        for x, y in val_dl:
            pred = head(backbone(x.to(device))).argmax(1)
            correct += (pred == y.to(device)).sum().item(); total += y.numel()
    print(f'ep={ep+1}  indoor_val_top1={correct/total:.3f}')

## 7. Loss — Logit Adjustment (Menon ICLR 2021)

Replaces the legacy *inverse-frequency CE × `HEALTHY_DOWNWEIGHT=0.25`* heuristic with

logits[y] -= τ · log π_y

with τ=1.0 and π_y = empirical class prior. Same protection against majority-class collapse, no hand-tuned downweight.

In [ ]:
import torch.nn.functional as F
from collections import Counter

class LogitAdjustmentLoss(nn.Module):
    def __init__(self, class_priors: torch.Tensor, tau: float = 1.0,
                 label_smoothing: float = 0.05):
        super().__init__()
        self.register_buffer('log_prior', torch.log(class_priors.clamp(min=1e-12)))
        self.tau = tau
        self.label_smoothing = label_smoothing

    def forward(self, logits, target):
        adjusted = logits - self.tau * self.log_prior
        return F.cross_entropy(adjusted, target, label_smoothing=self.label_smoothing)

def class_priors_from_imagefolder(ds) -> torch.Tensor:
    counts = Counter([y for _, y in ds.samples])
    pri = torch.tensor([counts.get(i, 0) for i in range(len(ds.classes))], dtype=torch.float32)
    return pri / pri.sum()

## 8. Indoor-targeted augmentations

Imitate the failure modes of indoor phone shots: warm/cool WB shifts, glare, defocus/motion blur, JPEG ringing, ISP smoothing, soft cast shadows.

**Disabled on purpose:** strong hue shifts and `Solarize` — colour itself is a signal of disease (chlorosis, browning, viral mottling). MixUp/CutMix kept low (0.10 / 0.15) for the same reason.

In [ ]:
from PIL import ImageFilter, ImageEnhance, Image as _Image
from io import BytesIO
import random

class WhiteBalanceShift:
    def __init__(self, p=0.5, strength=(0.8, 1.2)):
        self.p = p; self.lo, self.hi = strength
    def __call__(self, img):
        if random.random() > self.p: return img
        r_scale = random.uniform(self.lo, self.hi)
        b_scale = random.uniform(self.lo, self.hi)
        r, g, b = img.split()
        r = r.point(lambda v: max(0, min(255, int(v * r_scale))))
        b = b.point(lambda v: max(0, min(255, int(v * b_scale))))
        return _Image.merge('RGB', (r, g, b))

class Glare:
    def __init__(self, p=0.25): self.p = p
    def __call__(self, img):
        if random.random() > self.p: return img
        return ImageEnhance.Brightness(img).enhance(random.uniform(1.05, 1.4))

class DefocusOrMotion:
    def __init__(self, p=0.3): self.p = p
    def __call__(self, img):
        if random.random() > self.p: return img
        if random.random() < 0.5:
            return img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.8, 2.4)))
        return img.filter(ImageFilter.BoxBlur(random.uniform(1.0, 2.5)))

class JpegRecompress:
    def __init__(self, p=0.4, q_range=(45, 90)):
        self.p = p; self.q_range = q_range
    def __call__(self, img):
        if random.random() > self.p: return img
        buf = BytesIO()
        img.save(buf, format='JPEG', quality=random.randint(*self.q_range))
        buf.seek(0)
        return _Image.open(buf).convert('RGB')

indoor_train_tx = T.Compose([
    T.Resize(int(256 * 1.15)),
    T.RandomResizedCrop(256, scale=(0.7, 1.0), ratio=(0.85, 1.15)),
    T.RandomHorizontalFlip(),
    WhiteBalanceShift(p=0.5),
    Glare(p=0.25),
    DefocusOrMotion(p=0.3),
    JpegRecompress(p=0.4),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.0),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.15, scale=(0.02, 0.1)),
])

## 9. Teacher — EfficientNetV2-S

`timm` checkpoint `tf_efficientnetv2_s.in21k_ft_in1k` (21 M params, 84.2 % ImageNet top-1, quantises better than any ViT we tested). **ConvNeXt-Tiny** is the accuracy challenger.

Recipe: AdamW, weight_decay=1e-4, label_smoothing=0.05, cosine + 1-epoch warmup, 256 px batch=128 bf16-mixed; head-only 3 epochs (lr=3e-3) → unfreeze last 2 stages 12–15 epochs (head lr=5e-4, backbone lr=5e-5); MixUp 0.10, CutMix 0.10–0.20; Logit Adjustment loss (τ=1.0).

In [ ]:
import timm
from torch.amp import autocast, GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

BATCH, IMG = 128, 256

train_ds = ImageFolder(str(TRAIN_ROOT), transform=indoor_train_tx)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=8,
                      pin_memory=True, drop_last=True)

# Three independent val slices.
val_indomain_ds, val_indomain_dl = make_eval_loader(VAL_INDOMAIN, IMG)
val_realworld_ds, val_realworld_dl = make_eval_loader(VAL_PRIMARY, IMG)
val_studio_ds, val_studio_dl       = make_eval_loader(VAL_STUDIO, IMG)
CLASS_NAMES = train_ds.classes

teacher = timm.create_model('tf_efficientnetv2_s.in21k_ft_in1k',
                            pretrained=True, num_classes=len(CLASSES)).to(device)
priors = class_priors_from_imagefolder(train_ds).to(device)
loss_fn = LogitAdjustmentLoss(priors, tau=1.0, label_smoothing=0.05).to(device)
scaler = GradScaler('cuda')

def step_eval(model, tag):
    """Log all three val slices each epoch. ``val_realworld`` is the
    primary metric — ``val_indomain`` drives early stopping only."""
    return {
        "indomain":  evaluate(model, val_indomain_dl,  CLASS_NAMES, device, f"{tag} val_indomain"),
        "realworld": evaluate(model, val_realworld_dl, CLASS_NAMES, device, f"{tag} val_realworld"),
        "studio":    evaluate(model, val_studio_dl,    CLASS_NAMES, device, f"{tag} val_studio"),
    }

# Phase A — head only (3 epochs, lr=3e-3)
for n, p in teacher.named_parameters():
    p.requires_grad_(n.startswith('classifier'))
opt = torch.optim.AdamW([p for p in teacher.parameters() if p.requires_grad],
                        lr=3e-3, weight_decay=1e-4)
warm = LinearLR(opt, start_factor=0.1, total_iters=len(train_dl))
cos  = CosineAnnealingLR(opt, T_max=3*len(train_dl) - len(train_dl))
sched = SequentialLR(opt, [warm, cos], milestones=[len(train_dl)])
for ep in range(3):
    teacher.train()
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad()
        with autocast('cuda', dtype=torch.bfloat16):
            loss = loss_fn(teacher(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    step_eval(teacher, f'[head-only ep={ep+1}]')

# Phase B — unfreeze last 2 stages (12–15 epochs) — early-stop on val_indomain
for n, p in teacher.named_parameters():
    p.requires_grad_(True)
head_params = [p for n, p in teacher.named_parameters() if n.startswith('classifier')]
back_params = [p for n, p in teacher.named_parameters() if not n.startswith('classifier')]
opt = torch.optim.AdamW([
    {'params': head_params, 'lr': 5e-4},
    {'params': back_params, 'lr': 5e-5},
], weight_decay=1e-4)
EPOCHS_B = 13
warm = LinearLR(opt, start_factor=0.1, total_iters=len(train_dl))
cos  = CosineAnnealingLR(opt, T_max=EPOCHS_B*len(train_dl) - len(train_dl))
sched = SequentialLR(opt, [warm, cos], milestones=[len(train_dl)])

best_indomain_f1, best_state, patience, bad = -1.0, None, 4, 0
for ep in range(EPOCHS_B):
    teacher.train()
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad()
        with autocast('cuda', dtype=torch.bfloat16):
            loss = loss_fn(teacher(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    metrics = step_eval(teacher, f'[finetune ep={ep+1}]')
    if metrics['indomain']['disease_macro_f1'] > best_indomain_f1:
        best_indomain_f1 = metrics['indomain']['disease_macro_f1']
        best_state = {k: v.detach().cpu().clone() for k, v in teacher.state_dict().items()}
        bad = 0
    else:
        bad += 1
        if bad >= patience:
            print(f'Early stopping at epoch {ep+1} (no val_indomain improvement for {patience} eps)')
            break

if best_state is not None:
    teacher.load_state_dict(best_state)
torch.save(teacher.state_dict(), 'models/teacher_v2s.pth')


## 10. Knowledge distillation → MobileNetV3-Large student

Production artefact = MobileNetV3-Large (5.5 M params) distilled from V2-S.

L = α · CE(z_s, y) + (1 − α) · τ² · KL(σ(z_t/τ) || σ(z_s/τ))

α = 0.35, τ = 3.5, ~12 epochs. Student input still 256 px.

In [ ]:
from torchvision import models

ALPHA, TAU, EPOCHS_KD = 0.35, 3.5, 12

student = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
in_f = student.classifier[-1].in_features
student.classifier[-1] = nn.Linear(in_f, len(CLASSES))
student = student.to(device)

teacher.eval()
for p in teacher.parameters(): p.requires_grad_(False)

opt = torch.optim.AdamW(student.parameters(), lr=5e-4, weight_decay=1e-4)
warm = LinearLR(opt, start_factor=0.1, total_iters=len(train_dl))
cos  = CosineAnnealingLR(opt, T_max=EPOCHS_KD*len(train_dl) - len(train_dl))
sched = SequentialLR(opt, [warm, cos], milestones=[len(train_dl)])
ce = nn.CrossEntropyLoss(label_smoothing=0.05)

for ep in range(EPOCHS_KD):
    student.train()
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad()
        with autocast('cuda', dtype=torch.bfloat16):
            with torch.no_grad():
                t_logits = teacher(x)
            s_logits = student(x)
            ce_loss = ce(s_logits, y)
            kl_loss = F.kl_div(
                F.log_softmax(s_logits / TAU, dim=1),
                F.softmax(t_logits / TAU, dim=1),
                reduction='batchmean',
            ) * (TAU * TAU)
            loss = ALPHA * ce_loss + (1 - ALPHA) * kl_loss
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    print(f'[KD] ep={ep+1} ce={ce_loss.item():.3f} kl={kl_loss.item():.3f}')

torch.save(student.state_dict(), 'models/classifier.pth')

## 11. ONNX export + **static** INT8 (QDQ) + MaxLogit + Conformal

Two steps that operate on the trained student.

**Static INT8 QDQ.** `quantize_dynamic` only touches MatMul weights — that's ~0.2 % of the parameter count for MobileNetV3-Large, so accuracy drift is small but throughput barely changes. We swap to **static** quantization (`onnxruntime.quantization.quantize_static`) with a calibration set of **500 real user photos** from `val_realworld`. ORT explicitly warns that quantized accuracy is not guaranteed without validation, so the cell below **re-evaluates the INT8 ONNX model on `val_realworld` and refuses to write the artefact if disease macro-F1 drops by more than 0.02 vs FP32**.

**Inductive conformal calibration.** Temperature scaling fixes miscalibration; conformal prediction adds an honest *prediction set* with a finite-sample coverage guarantee (Vovk; Angelopoulos & Bates 2021). We calibrate on a 1–2 k user-photo holdout and persist the artefact to `models/conformal.json`. Production reads it via `server/conformal.py` — no model changes.

**MaxLogit OOD floor.** Sweep on a held-out OOD bag (OE-pool tail) to hit the target FPR@95-TPR; persist into `classes.json` under `ood.max_logit_floor`.


In [ ]:
import json, numpy as np, torch
from pathlib import Path
import onnxruntime as ort
from onnxruntime.quantization import (
    quantize_static, CalibrationDataReader, QuantFormat, QuantType,
)

student.eval()
dummy = torch.randn(1, 3, IMG, IMG, device=device)

# 1. ONNX FP32 export.
torch.onnx.export(student, dummy, 'models/classifier.onnx', opset_version=18,
                  input_names=['input'], output_names=['logits'],
                  dynamic_axes={'input': {0: 'N'}, 'logits': {0: 'N'}})

# 2. Static INT8 calibration — 500 real user-photo crops from val_realworld.
class RealWorldCalibrationReader(CalibrationDataReader):
    """Streams 500 calibration tensors through ORT in eval-transform space."""
    def __init__(self, dataset, n=500, input_name='input'):
        self.input_name = input_name
        idx = np.random.RandomState(0).permutation(len(dataset))[:n]
        self._iter = iter([
            (dataset[i][0].unsqueeze(0).numpy().astype(np.float32),) for i in idx
        ])
    def get_next(self):
        nxt = next(self._iter, None)
        return None if nxt is None else {self.input_name: nxt[0]}

quantize_static(
    model_input='models/classifier.onnx',
    model_output='models/classifier_int8.onnx',
    calibration_data_reader=RealWorldCalibrationReader(val_realworld_ds, n=500),
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QInt8,
    weight_type=QuantType.QInt8,
    per_channel=True,
)

# 3. **Mandatory** post-quantization validation on val_realworld. ORT
#    does not guarantee accuracy after static quantization — we either
#    confirm the drift is within tolerance or fail loudly.
def eval_onnx(path, dl, class_names):
    sess = ort.InferenceSession(path, providers=['CPUExecutionProvider'])
    in_name = sess.get_inputs()[0].name
    y_true, y_pred = [], []
    for x, y in dl:
        logits = sess.run(None, {in_name: x.numpy().astype(np.float32)})[0]
        y_pred.extend(logits.argmax(1).tolist()); y_true.extend(y.tolist())
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    disease_idx = [i for i, n in enumerate(class_names)
                   if n not in ('healthy', 'not_a_plant')]
    from sklearn.metrics import f1_score
    return float(f1_score(y_true, y_pred, labels=disease_idx,
                          average='macro', zero_division=0))

f1_fp32 = eval_onnx('models/classifier.onnx',      val_realworld_dl, CLASS_NAMES)
f1_int8 = eval_onnx('models/classifier_int8.onnx', val_realworld_dl, CLASS_NAMES)
drift = f1_fp32 - f1_int8
print(f'val_realworld disease F1  fp32={f1_fp32:.4f}  int8={f1_int8:.4f}  drift={drift:+.4f}')
assert drift <= 0.02, (
    f'Static INT8 quantization dropped disease F1 by {drift:.3f} (>0.02). '
    f'Re-run with a different calibration set or fall back to FP32.'
)

# 4. Temperature scaling on val_realworld NLL.
import torch.nn.functional as F
logits_val, y_val = [], []
with torch.inference_mode():
    for x, y in val_realworld_dl:
        logits_val.append(student(x.to(device)).cpu()); y_val.append(y)
logits_val = torch.cat(logits_val); y_val = torch.cat(y_val)

T_param = torch.tensor(1.0, requires_grad=True)
opt = torch.optim.LBFGS([T_param], lr=0.05, max_iter=50)
def closure():
    opt.zero_grad()
    loss = F.cross_entropy(logits_val / T_param.clamp(min=0.05), y_val)
    loss.backward(); return loss
opt.step(closure)
T_val = float(T_param.detach())
json.dump({'temperature': T_val}, open('models/temperature.json', 'w'), indent=2)
print(f'fitted T={T_val:.3f}')

id_max_logit = logits_val.max(dim=1).values.numpy()
print(f'in-dist MaxLogit  mean={id_max_logit.mean():.2f}  '
      f'p10={np.percentile(id_max_logit, 10):.2f}')


## 11a. Conformal calibration — RAPS prediction sets at 90 % coverage

Calibrate on a 1–2 k user-photo holdout (a separate slice from `val_realworld`; no overlap with the 500 calibration photos used for static INT8). One run produces `models/conformal.json` consumed by the production pipeline; no online re-training.

We default to **RAPS** (Angelopoulos 2021) over plain APS because the size penalty avoids pathologically large sets on the hard "not really visible at 256 px" examples that dominate the long tail. `lambda_reg=0.01`, `k_reg=1`, `alpha=0.1` (90 % coverage) are sensible defaults; tune `alpha` per release. Empirical coverage on the calibration set itself should match `1-alpha` within Monte-Carlo noise (≈±2 % at n=1500).

In [ ]:
import sys, pathlib, numpy as np, torch, json
sys.path.insert(0, str(pathlib.Path('.').resolve().parent / 'server'))
from conformal import calibrate_q_hat

# Held-out conformal cal-set: a separate 1-2k user-photo slice. Must NOT
# overlap with the 500 photos used as the INT8 calibration reader.
CAL_ROOT = ROOT / 'val' / 'conformal_cal_v1'
cal_ds, cal_dl = make_eval_loader(CAL_ROOT, IMG)

logits_cal, y_cal = [], []
with torch.inference_mode():
    for x, y in cal_dl:
        logits_cal.append(student(x.to(device)).cpu()); y_cal.append(y)
logits_cal = torch.cat(logits_cal)
y_cal = torch.cat(y_cal).numpy()

# Apply the temperature fitted in the previous cell so conformal sees
# the same calibrated softmax that production will see.
probs_cal = torch.softmax(logits_cal / max(T_val, 1e-3), dim=1).numpy()

cfg = calibrate_q_hat(
    probs_cal, y_cal,
    method='raps', alpha=0.10,
    lambda_reg=0.01, k_reg=1,
)
out = {
    'method': cfg.method,
    'alpha': cfg.alpha,
    'q_hat': cfg.q_hat,
    'lambda_reg': cfg.lambda_reg,
    'k_reg': cfg.k_reg,
    'calibration_n': cfg.calibration_n,
    'empirical_coverage': cfg.empirical_coverage,
}
json.dump(out, open('models/conformal.json', 'w'), indent=2)
print('conformal:', out)


## 12. Evaluation contract

Three independent slices, three reports per release. Logged every epoch during training and replayed once on the final ONNX INT8 artefact:

| Slice | Role | What we look at | Pass criterion |
|---|---|---|---|
| `val_indomain` | training-loop metric | macro-F1 over diseases — drives early stopping | non-decreasing for ≥ N epochs |
| `val_realworld` | **primary — ship/no-ship gate** | macro-F1 over diseases, per-class recall, healthy-bias, confusion matrix | macro-F1 ∈ **0.70 – 0.78** target band; no rare class with recall = 0; healthy-bias < 1.5 |
| `val_studio` | transfer / background-bias monitor | top-1 + Noyan's 8-px-background sanity check | tracked, never blocking |

Top-1 alone hides two failure modes the per-class recall surfaces immediately: a healthy-class collapse (model lazily routes everything to `healthy` because it dominates the train set) and the death of a rare class (`edema`, `viral_disease` going to recall 0). Both are pre-existing strengths of the legacy pipeline that we keep — disease macro-F1 as the primary metric, healthy-bias monitor each epoch, group-aware splits by `plant_id`, hardlinks for image stores, sparse checkout for big repos, ONNX sanity drift-check after every quantization.

A v2.x model that gains primary F1 but tanks `val_studio` top-1 is fine — that means we're learning leaves, not backgrounds (Noyan 2022 baseline = 49 % from 8 background pixels).

### Optional: Coreset / dataset pruning

Once the v2.0 weights are landed, **DeepCore** (Guo et al. 2022) is the cheapest place to look for further gains if iteration time becomes a bottleneck. Random selection remains a strong baseline — don't skip it as the no-op control. **Do not** invest in dataset distillation: at our resolution and with this many ambiguous label boundaries the published ROI is weak. Model pruning is also low priority — KD into MobileNetV3-Large + static INT8 already covers 95 % of the runtime budget.
